In [1]:
!pip install -q chromadb transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [2]:
import os
import re
import torch
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [3]:
EMBED_model = "Omartificial-Intelligence-Space/Arabic-Triplet-Matryoshka-V2"
LLM_model = "humain-ai/ALLaM-7B-Instruct-preview"

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_model,
    #quantization_config=bnb_config,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
tokenizer = AutoTokenizer.from_pretrained(LLM_model)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [6]:
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/637 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [7]:
def split_poem_blocks(text):
    chunks = re.split(r"\n[-]{10,}\n", text)

    return [chunk.strip() for chunk in chunks if chunk.strip()]

# ------------------------------------------------------------------------

def load_files(folder_path):
    all_chunks = []
    metadatas = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = split_poem_blocks(text)

            for i, chunk in enumerate(chunks):
                all_chunks.append(chunk)
                metadatas.append({
                    "source": filename,
                    "chunk_id": i
                })

    return all_chunks, metadatas

# ------------------------------------------------------------------------

DATA_FOLDER = "datasets"

chunks, metadatas = load_files(DATA_FOLDER)

print(f"chunks: {len(chunks)}")

chunks: 758


In [8]:
client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection(name="poetry", embedding_function=embedding_func)

BATCH_SIZE = 64

for i in range(0, len(chunks), BATCH_SIZE):

    batch_chunks = chunks[i:i+BATCH_SIZE]
    batch_metadatas = metadatas[i:i+BATCH_SIZE]
    batch_ids = [str(x) for x in range(i, i + len(batch_chunks))]

    collection.add(
        documents = batch_chunks,
        metadatas = batch_metadatas,
        ids=batch_ids
    )

In [9]:
def search(query, n_results=1):

    results = collection.query(
        query_texts= [query],
        n_results=n_results
    )

    docs = results["documents"][0]

    return docs

In [35]:
def ask_rag(query):

    docs = search(query)

    messages = [
        {
            "role": "system",
            "content": "أنت مساعد متخصص في الشعر العربي. أجب بشكل مباشر على السؤال اعتمادًا على المعلومات المعطاة فقط."
        },
        {
            "role": "user",
            "content": f"""

    النص المرجعي:
    {docs}

    السؤال:
    {query}
    """
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt", return_token_type_ids=False)
    inputs = {k: v.to('cuda') for k,v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.1,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    answer = answer.replace("▁", " ")

    print(answer)

In [38]:
ask_rag(" ما اعراب هل غادر الشعراء من متردم")

 ​ هل :  حرف  استفهام. غ اد َرَ :  فعل  ماض ،  وفاع له  ضمير  مست تر  تقديره  " هو ". الش عراء َ :  مفعول  به. م ِنْ :  حرف  جر. م تر دم ٍ :  اسم  مجر ور  ب ـ" م ِنْ "،  وعل امة  ج ره  الكس رة  المقدرة  على  آخره  منع  من  ظهورها  اشت غال  المحل  بحركة  حرف  الجر  الزائد.  


In [39]:
from openai import OpenAI

client_llm = OpenAI(
    api_key="sk-GDsDnrnoU68uCBi2jhfC0A",
    base_url="https://elmodels.ngrok.app/v1"
)

def ask_rag_nuha(query):

    docs = search(query)

    context = "\n\n".join(docs)

    response = client_llm.chat.completions.create(
        model="nuha-2.0",
        messages = [
                {
                "role": "system",
                "content": "أنت مساعد متخصص في إعراب وشرح الشعر العربي. أجب بشكل مباشر اعتمادًا على المعلومات المعطاة فقط."
            },
            {
                "role": "user",
                "content": f"""
    النص المرجعي:
    {context}

    السؤال:
    {query}
    """
            }
        ]
    )

    response = response.choices[0].message.content
    return response

In [41]:
print(ask_rag_nuha(" ما اعراب هل غادر الشعراء من متردم"))

بناءً على النص المرجعي المقدم، إعراب جملة "هل غادر الشعراء من متردم" هو كالتالي:

*   **هل**: حرف استفهام.
*   **غادر**: فعل ماضٍ.
*   **الشعراء**: فاعل مرفوع (وعلامة رفعه الضمة المقدرة على الألف لأنه جمع مؤنث سالم).
*   **من**: حرف جر زائد.
*   **متردم**: مفعول به منصوب، وعلامة نصبه الفتحة المقدرة على آخره، منع من ظهورها اشتغال المحل بحركة حرف الجر الزائد.


In [44]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 11.1 MB/s eta 0:00:00


In [45]:
from groq import Groq
client_g = Groq(api_key="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")

In [53]:
def ask_rag_g(query):
    docs = search(query)
    context = "\n\n".join(docs)

    prompt = f"""
النص المرجعي:
{context}

السؤال:
{query}
"""

    response = client_g.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "أنت مساعد متخصص في إعراب وشرح الشعر العربي. أجب على السؤال اعتمادًا على المعلومات المعطاة فقط."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    raw_output = response.choices[0].message.content.strip()

    print(raw_output)

In [54]:
print(ask_rag_g(" ما معنى هل غادر الشعراء من متردم"))

المعنى: هل ترك الشعراء فنًا من فنون الشعر لم يسلكوه.
None
